# Project: managing a research programme

Check and summarize the participants of a programme of 50 projects, with the standard module `csv`, and write Python formulas in Grist.

The full explanations are in [research_programme.md](research_programme.md). *This work is licensed under [Creative Commons Attribution 4.0 International License](http://creativecommons.org/licenses/by/4.0/).*

## How to use this notebook

* Run the cells **in order**, with `Shift+Enter`.
* Cells starting with `%%writefile` **create a file** (module, tests or script)
  in the folder of this notebook: like in the lesson, the project is made of
  real `.py` files, that you can also open in a text editor or Spyder.
* `%run script.py` runs a script as from the shell, and the cell with
  `pytest` runs the tests.
* Start JupyterLab from an **empty folder** dedicated to this project, in the
  `sci` environment:

  ```
  (sci) $ mkdir my_project
  (sci) $ cd my_project
  (sci) $ jupyter lab
  ```

## Some data

This script creates a **fake** list of participants, with duplicated registrations and missing emails.

In [ ]:
%%writefile generate_participants.py
"""Generate a fake list of participants of a research programme."""

import csv
import unicodedata

import numpy as np

rng = np.random.default_rng(seed=7)

first_names = ["Alice", "Bruno", "Chloé", "David", "Emma", "Farid", "Gaëlle",
               "Hugo", "Inès", "Julien", "Karima", "Louis", "Manon", "Nicolas",
               "Océane", "Pierre", "Quentin", "Rose", "Samir", "Théo"]
last_names = ["Martin", "Bernard", "Dubois", "Thomas", "Robert", "Richard",
              "Petit", "Durand", "Leroy", "Moreau", "Simon", "Laurent",
              "Lefebvre", "Michel", "Garcia", "David", "Bertrand", "Roux"]
institutes = ["INC", "INSB", "INP", "INSU", "INEE", "INSMI", "IN2P3", "INSIS"]
roles = ["PI", "Researcher", "Engineer", "PhD student", "Postdoc"]

rows = []
for i in range(1000):
    first = str(rng.choice(first_names))
    last = str(rng.choice(last_names))
    ascii_name = unicodedata.normalize("NFKD", f"{first}.{last}").encode("ascii", "ignore").decode()
    email = f"{ascii_name.lower()}{i}@example.fr"
    if rng.random() < 0.02:
        email = ""                                  # missing email
    rows.append({
        "first_name": first,
        "last_name": last,
        "email": email,
        "institute": str(rng.choice(institutes, p=[0.25, 0.2, 0.15, 0.1, 0.1, 0.05, 0.1, 0.05])),
        "role": str(rng.choice(roles, p=[0.1, 0.35, 0.2, 0.2, 0.15])),
        "project": f"P{rng.integers(1, 51):02d}",
    })

# a few people registered twice, with different capitalisation
for k in range(5):
    duplicate = dict(rows[k * 100])
    duplicate["email"] = duplicate["email"].upper()
    rows.append(duplicate)

with open("participants.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)
print("Written participants.csv with", len(rows), "rows")

In [ ]:
%run generate_participants.py

## Step 1: reading and counting

`csv.DictReader` returns each row as a dictionary. In `count_by`,
`counts.get(value, 0) + 1` adds one to the count, whether the value has
already been seen or not.

In [ ]:
%%writefile programme.py
"""Tools to analyse the participants of a research programme."""

import csv


def load_participants(filename):
    """Read a CSV file and return a list of dictionaries, one per participant."""
    with open(filename, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))


def count_by(participants, column):
    """Count the participants for each value of a column.

    Parameters
    ----------
    participants : list of dict
        The participants, as returned by load_participants.
    column : str
        Name of the column, e.g. "project" or "institute".

    Returns
    -------
    dict
        Value of the column -> number of participants, sorted by decreasing count.

    Examples
    --------
    >>> count_by([{"role": "PI"}, {"role": "Engineer"}, {"role": "PI"}], "role")
    {'PI': 2, 'Engineer': 1}
    """
    counts = {}
    for participant in participants:
        value = participant[column]
        counts[value] = counts.get(value, 0) + 1
    sorted_counts = {}
    for value in sorted(counts, key=counts.get, reverse=True):
        sorted_counts[value] = counts[value]
    return sorted_counts


def normalize_email(email):
    """Return the email in a canonical form (no spaces, lower case)."""
    return email.strip().lower()


def find_duplicates(participants):
    """Return the emails appearing more than once (ignoring case and spaces)."""
    seen = set()
    duplicates = []
    for participant in participants:
        email = normalize_email(participant["email"])
        if email == "":
            continue
        if email in seen and email not in duplicates:
            duplicates.append(email)
        seen.add(email)
    return duplicates


def find_missing(participants, column):
    """Return the participants for which a column is empty."""
    return [p for p in participants if p[column].strip() == ""]

In [ ]:
from programme import load_participants, count_by

participants = load_participants("participants.csv")
print(len(participants))
print(participants[0])
count_by(participants, "role")

## Step 2: tests

In [ ]:
%%writefile test_programme.py
"""Tests for programme.py."""

from programme import count_by, find_duplicates, find_missing, normalize_email

PEOPLE = [
    {"email": "a@x.fr", "project": "P01"},
    {"email": " A@X.fr ", "project": "P02"},
    {"email": "b@x.fr", "project": "P01"},
    {"email": "", "project": "P03"},
]


def test_count_by():
    assert count_by(PEOPLE, "project") == {"P01": 2, "P02": 1, "P03": 1}


def test_normalize_email():
    assert normalize_email("  Alice.Martin@CNRS.fr ") == "alice.martin@cnrs.fr"


def test_find_duplicates_ignores_case_and_spaces():
    assert find_duplicates(PEOPLE) == ["a@x.fr"]


def test_empty_emails_are_not_duplicates():
    assert find_duplicates([{"email": ""}, {"email": ""}]) == []


def test_find_missing():
    assert find_missing(PEOPLE, "email") == [{"email": "", "project": "P03"}]

In [ ]:
import sys
!{sys.executable} -m pytest -q --doctest-modules programme.py test_programme.py

## Step 3: the report

In [ ]:
%%writefile report.py
"""Produce a report on the participants of the programme."""

import argparse
import csv

import matplotlib.pyplot as plt

from programme import count_by, find_duplicates, find_missing, load_participants


def save_counts(counts, filename, column):
    """Save a dictionary of counts as a CSV file."""
    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([column, "participants"])
        for value, count in counts.items():
            writer.writerow([value, count])


def main():
    parser = argparse.ArgumentParser(description="Programme report.")
    parser.add_argument("filename", help="CSV file of participants")
    args = parser.parse_args()

    participants = load_participants(args.filename)
    print(f"{len(participants)} rows")

    duplicates = find_duplicates(participants)
    print(f"{len(duplicates)} duplicated emails:", ", ".join(duplicates))
    missing = find_missing(participants, "email")
    print(f"{len(missing)} participants without email, saved in missing_emails.csv")
    with open("missing_emails.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(participants[0]))
        writer.writeheader()
        writer.writerows(missing)

    by_project = count_by(participants, "project")
    by_institute = count_by(participants, "institute")
    save_counts(by_project, "by_project.csv", "project")
    save_counts(by_institute, "by_institute.csv", "institute")
    print("Largest projects:", list(by_project.items())[:3])

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].bar(list(by_project), list(by_project.values()))
    axes[0].set_title("Participants per project")
    axes[0].tick_params(axis="x", labelrotation=90, labelsize=6)
    axes[1].barh(list(by_institute), list(by_institute.values()))
    axes[1].set_title("Participants per institute")
    axes[1].invert_yaxis()
    fig.tight_layout()
    fig.savefig("report.png", dpi=150)
    plt.show()


if __name__ == "__main__":
    main()

In [ ]:
%run report.py participants.csv

## Python in Grist

Grist formulas are written in Python. `$column` refers to a column of the
same row:

```python
$first_name + " " + $last_name.upper()
$email.strip().lower()
len(Participants.lookupRecords(project=$name))
```

See the lesson for multi-line formulas.

## Going further

**1.** Write `count_by_two(participants, column1, column2)` counting the participants for each pair of values, e.g. (project, role).

<details><summary>Answer</summary>

```python
def count_by_two(participants, column1, column2):
    counts = {}
    for participant in participants:
        key = (participant[column1], participant[column2])
        counts[key] = counts.get(key, 0) + 1
    return counts
```

</details>

In [ ]:
# Your code here

**2.** Write a function returning the projects without PI.

<details><summary>Answer</summary>

```python
def projects_without_pi(participants):
    all_projects = set()
    with_pi = set()
    for participant in participants:
        all_projects.add(participant["project"])
        if participant["role"] == "PI":
            with_pi.add(participant["project"])
    return sorted(all_projects - with_pi)
```

</details>

In [ ]:
# Your code here